## BPE- Byte-Pair-Encoding

In [1]:
import re
from collections import Counter

In [2]:
text= "set new new renew reset renew"

def spliting(x):
    token_word =re.findall(r"\s?\w+",x)
    x=([s.replace(" ","_") for s in token_word])
    return x

token_words=spliting(text)
print(token_words)

['set', '_new', '_new', '_renew', '_reset', '_renew']


#### vocabularies0 {_,e,n,r,s,t, w }

In [4]:
def get_vocabulary(token_words):
    ch=set()
    for x in token_words:
        ch.update(re.findall(r"\w",x))
    return(ch)

In [5]:
# another way use char-by-char useing for-loop
def get_vocabulary2(token_words):
    listofchar=[]
    for x in token_words:
        for ch in x:
            if ch not in listofchar:
                listofchar.append(ch)
    return(ch)
v2=get_vocabulary(token_words)
print("vocabularies 0 ", v2)

vocabularies 0  {'w', 't', 's', 'r', '_', 'n', 'e'}


In [6]:
def couting_frequnacy_token(token_word):
    di=dict()
    frequency_counts= Counter(token_word)
    for w,f in frequency_counts.items():
        new_w =re.findall(r"\w",w)
        di[tuple(new_w)]=f
    return di
words_freq=couting_frequnacy_token(token_words)
print(words_freq)

{('s', 'e', 't'): 1, ('_', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 's', 'e', 't'): 1}


In [7]:
def couting_frequnacy_token(token_word):
    di=dict()
    frequency_counts= Counter(token_word)
    for w,f in frequency_counts.items():
        new_w =tuple(w)    #another way to convert string to tuple
        di[new_w]=f
    return di
words_freq=couting_frequnacy_token(token_words)
print(words_freq)

{('s', 'e', 't'): 1, ('_', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 's', 'e', 't'): 1}


The loop handles it, because it adds `f` **at every position where the pair appears**, not once per word.

`banana` → `('b','a','n','a','n','a')`, with `f = 1`:

| i | pair | add | running total for (a, n) |
|---|---|---|---|
| 0 | (b, a) | +1 | 0 |
| 1 | (a, n) | +1 | **1** |
| 2 | (n, a) | +1 | 1 |
| 3 | (a, n) | +1 | **2** |
| 4 | (n, a) | +1 | 2 |

The loop meets `(a, n)` twice, at i=1 and i=3, and adds 1 each time, so the total is **2**.

So the count is (times the pair appears in the word) × `f`, and the loop does that multiplication for you just by visiting every position.

In [8]:
def count_pair_freq(tokens:dict):
    pairs=dict()
    for w,f in tokens.items():
        for i in range(0,len(w)-1):
            pair = (w[i],w[i+1])
            pairs[pair]=pairs.get(pair, 0)+f
          
    return pairs


In [9]:
print(words_freq)

{('s', 'e', 't'): 1, ('_', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 'n', 'e', 'w'): 2, ('_', 'r', 'e', 's', 'e', 't'): 1}


In [10]:
def marge(words_freq:dict,best_pair):
    new_words_freq=dict()
    l,r= best_pair
    marge_char=l+r
    for w,f in words_freq.items():
        new_word = []
        i = 0
        while i< len(w):
            if i<len(w)-1 and w[i] == l and w[i+1]==r:
               new_word.append(marge_char) 
               i=i+2 
            else:
                new_word.append(w[i])
                i=i+1

        new_words_freq [tuple(new_word)]=f
    return new_words_freq

In [11]:
def fmt_word(w):
    return " ".join(f"[{t}]" for t in w)

def show_step(step, pairs, best_pair, words_freq, vocab):
    print(f"\n{'='*20} STEP {step} {'='*20}")

    print("\nPair counts (highest first):")
    for p, c in sorted(pairs.items(), key=lambda x: -x[1]):
        mark = "  <-- best" if p == best_pair else ""
        print(f"   {p[0]:>6} + {p[1]:<6} : {c}{mark}")

    print(f"\nMerge: '{best_pair[0]}' + '{best_pair[1]}' -> '{best_pair[0] + best_pair[1]}'")

    print("\nWords after merge:")
    for w, f in words_freq.items():
        print(f"   {f} x  {fmt_word(w)}")

    print(f"\nVocab ({len(vocab)}): {sorted(vocab, key=len)}")

In [12]:
def train(words_freq, vocab, k):
    merges = list()
    for i in range(k):
        pairs = count_pair_freq(words_freq)
        if len(pairs)==0:
            print("not found pair")
            break
        best_pair=max(pairs.items(),key=lambda x:x[1])[0]
        merges.append(best_pair)
        words_freq = marge(words_freq, best_pair)
        vocab.add(best_pair[0] + best_pair[1])
        show_step(i + 1, pairs, best_pair, words_freq, vocab)

    return merges,vocab

start = get_vocabulary(token_words)
merges, vocab = train(words_freq, start.copy(), 8)

print(merges)
print(f"\n{'#'*20} FINAL {'#'*20}")
print(f"Start vocab ({len(start)}): {sorted(start)}")
print("\nMerges in order:")
for n, (l, r) in enumerate(merges, 1):
    print(f"   {n}. {l} + {r} -> {l + r}")
print(f"\nFinal vocab ({len(vocab)}): {sorted(vocab, key=len)}")



==================== STEP 1 ====================

Pair counts (highest first):
        n + e      : 4  <-- best
        e + w      : 4
        _ + r      : 3
        r + e      : 3
        s + e      : 2
        e + t      : 2
        _ + n      : 2
        e + n      : 2
        e + s      : 1

Merge: 'n' + 'e' -> 'ne'

Words after merge:
   1 x  [s] [e] [t]
   2 x  [_] [ne] [w]
   2 x  [_] [r] [e] [ne] [w]
   1 x  [_] [r] [e] [s] [e] [t]

Vocab (8): ['r', 'w', 't', '_', 'n', 'e', 's', 'ne']

==================== STEP 2 ====================

Pair counts (highest first):
       ne + w      : 4  <-- best
        _ + r      : 3
        r + e      : 3
        s + e      : 2
        e + t      : 2
        _ + ne     : 2
        e + ne     : 2
        e + s      : 1

Merge: 'ne' + 'w' -> 'new'

Words after merge:
   1 x  [s] [e] [t]
   2 x  [_] [new]
   2 x  [_] [r] [e] [new]
   1 x  [_] [r] [e] [s] [e] [t]

Vocab (9): ['s', 'r', 'e', 'w', 't', '_', 'n', 'ne', 'new']

==================== 

In [13]:
def merge_word(word, pair):
    l, r = pair
    new_word = []
    i = 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == l and word[i + 1] == r:
            new_word.append(l + r)
            i = i + 2
        else:
            new_word.append(word[i])
            i = i + 1
    return tuple(new_word)


def encode_word(word, merges):
    tokens = tuple(word)                  # step 1: split into characters
    for pair in merges:
        new_tokens = merge_word(tokens, pair)
        if new_tokens != tokens:
            print(f"   {pair[0]} + {pair[1]}  ->  {new_tokens}")
        tokens = new_tokens
    return tokens

def encode_text(text, merges):
    words = text.split()
    all_tokens = []
    for i, w in enumerate(words):
        if i > 0:                         # same rule as training:
            w = "_" + w                   # every word after the first gets "_"
        all_tokens.extend(encode_word(w, merges))
    return all_tokens

In [14]:
print(encode_text("set new renewal anew reset", merges))

   s + e  ->  ('se', 't')
   se + t  ->  ('set',)
   n + e  ->  ('_', 'ne', 'w')
   ne + w  ->  ('_', 'new')
   _ + new  ->  ('_new',)
   n + e  ->  ('_', 'r', 'e', 'ne', 'w', 'a', 'l')
   ne + w  ->  ('_', 'r', 'e', 'new', 'a', 'l')
   _ + r  ->  ('_r', 'e', 'new', 'a', 'l')
   _r + e  ->  ('_re', 'new', 'a', 'l')
   _re + new  ->  ('_renew', 'a', 'l')
   n + e  ->  ('_', 'a', 'ne', 'w')
   ne + w  ->  ('_', 'a', 'new')
   _ + r  ->  ('_r', 'e', 's', 'e', 't')
   _r + e  ->  ('_re', 's', 'e', 't')
   s + e  ->  ('_re', 'se', 't')
   se + t  ->  ('_re', 'set')
['set', '_new', '_renew', 'a', 'l', '_', 'a', 'new', '_re', 'set']
